# 📊 Notebook 04 — Gold Aggregations & KPI Reporting

**Goal:** Build pre-aggregated Gold tables and KPI views optimized for Power BI Direct Lake.

> **Run time:** ~5 min

These Gold tables power the executive dashboard — branch performance, product profitability, customer segment analysis, and monthly trends.

In [ ]:
# Import Spark SQL functions used to calculate gold-layer KPI aggregations.
from pyspark.sql import functions as F

# Load the gold fact tables that will feed branch, customer, product, and monthly KPI summaries.
fact_txn   = spark.table('fact_transactions')
fact_loans = spark.table('fact_loans')
# Confirm the fact tables are loaded before building downstream gold aggregates.
print('Loaded fact tables')

## Gold Table 1: Branch Performance KPIs

In [ ]:
# Aggregate fact_loans by branch to calculate loan volume, pricing, default, and outstanding balance KPIs.
# This creates the branch performance mart used to compare regional lending performance.
gold_branch_perf = fact_loans.groupBy('BranchID','BranchName','Region') \
    .agg(
        F.count('LoanID').alias('TotalLoans'),
        F.round(F.sum('LoanAmount'), 0).alias('TotalLoanVolume'),
        F.round(F.avg('LoanAmount'), 0).alias('AvgLoanSize'),
        F.round(F.avg('InterestRate'), 2).alias('AvgInterestRate'),
        F.sum('IsDefault').alias('TotalDefaults'),
        F.round(F.sum('IsDefault') * 100.0 / F.count('LoanID'), 1).alias('DefaultRate_Pct'),
        F.round(F.sum('OutstandingBalance'), 0).alias('TotalOutstanding')
    )

# Write gold_branch_performance to Delta so branch-level KPI reporting can query a persisted aggregate table.
gold_branch_perf.write.format('delta').mode('overwrite').saveAsTable('gold_branch_performance')
# Print the branch performance row count to validate the aggregation completed.
print(f'gold_branch_performance: {gold_branch_perf.count()} rows')
# Preview the highest-volume branches to confirm the branch KPI table looks correct.
gold_branch_perf.orderBy('TotalLoanVolume', ascending=False).show()

## Gold Table 2: Monthly Transaction Trends

In [ ]:
# Aggregate fact_transactions by calendar month to calculate transaction counts, amounts, activity, and failure KPIs.
# This monthly trend table supports time-series analysis of customer and account activity.
gold_monthly = fact_txn.filter(F.col('Year').isNotNull()) \
    .groupBy('Year','MonthName','Month','Quarter') \
    .agg(
        F.count('TransactionID').alias('NumTransactions'),
        F.round(F.sum('Amount'), 0).alias('TotalAmount'),
        F.round(F.avg('Amount'), 2).alias('AvgTransactionAmount'),
        F.countDistinct('CustomerID').alias('UniqueCustomers'),
        F.countDistinct('AccountID').alias('ActiveAccounts'),
        F.sum(F.when(F.col('Status') == 'Failed', 1).otherwise(0)).alias('FailedTransactions'),
        F.sum(F.when(F.col('IsLargeTransaction') == True, F.col('Amount')).otherwise(0)).alias('LargeTransactionVolume')
    ) \
    .orderBy('Year','Month')

# Write gold_monthly_trends to Delta so monthly transaction KPIs are available for dashboards.
gold_monthly.write.format('delta').mode('overwrite').saveAsTable('gold_monthly_trends')
# Print the monthly trend row count to validate the aggregation completed.
print(f'gold_monthly_trends: {gold_monthly.count()} rows')
# Preview monthly transaction KPIs to confirm the derived measures look correct.
gold_monthly.show()

## Gold Table 3: Customer Segment Analysis

In [ ]:
# Aggregate fact_loans by customer segment, credit tier, and age group to profile lending behavior and risk.
# This customer segment mart supports portfolio analysis across demographic and credit cohorts.
gold_segments = fact_loans.groupBy('CustomerSegment','CreditScoreTier','AgeGroup') \
    .agg(
        F.count('LoanID').alias('NumLoans'),
        F.countDistinct('CustomerID').alias('UniqueCustomers'),
        F.round(F.avg('LoanAmount'), 0).alias('AvgLoanAmount'),
        F.round(F.avg('InterestRate'), 2).alias('AvgInterestRate'),
        F.round(F.sum('IsDefault') * 100.0 / F.count('LoanID'), 1).alias('DefaultRate_Pct'),
        F.round(F.sum('LoanAmount'), 0).alias('TotalLoanVolume')
    )

# Write gold_customer_segments to Delta so segment-level KPIs can be queried efficiently.
gold_segments.write.format('delta').mode('overwrite').saveAsTable('gold_customer_segments')
# Print the customer segment row count to validate the aggregation completed.
print(f'gold_customer_segments: {gold_segments.count()} rows')
# Preview the highest-volume customer segments to confirm the KPI table looks correct.
gold_segments.orderBy('TotalLoanVolume', ascending=False).show()

## Gold Table 4: Product Performance

In [ ]:
# Aggregate fact_loans by product to calculate loan volume, pricing, defaults, and outstanding balance KPIs.
# This product performance mart supports comparisons across loan products in the gold layer.
gold_products = fact_loans.groupBy('ProductID','ProductName','ProductType') \
    .agg(
        F.count('LoanID').alias('NumLoans'),
        F.round(F.sum('LoanAmount'), 0).alias('TotalLoanVolume'),
        F.round(F.avg('LoanAmount'), 0).alias('AvgLoanAmount'),
        F.round(F.avg('InterestRate'), 2).alias('AvgInterestRate'),
        F.round(F.sum('OutstandingBalance'), 0).alias('TotalOutstanding'),
        F.sum('IsDefault').alias('Defaults'),
        F.round(F.sum('IsDefault') * 100.0 / F.count('LoanID'), 1).alias('DefaultRate_Pct')
    )

# Write gold_product_performance to Delta so product-level KPI reporting can reuse the aggregate table.
gold_products.write.format('delta').mode('overwrite').saveAsTable('gold_product_performance')
# Print the product performance row count to validate the aggregation completed.
print(f'gold_product_performance: {gold_products.count()} rows')
# Preview the top loan products to confirm the KPI table looks correct.
gold_products.orderBy('TotalLoanVolume', ascending=False).show()

## Executive Summary Query — Cross all Gold Tables

In [ ]:
%%sql
-- Top 5 branches by loan volume with default rate
-- Show the highest-volume branches with default and outstanding balances for a quick executive summary.
SELECT
    BranchName,
    Region,
    TotalLoans,
    CONCAT('$', FORMAT_NUMBER(TotalLoanVolume, 0))  AS LoanVolume,
    CONCAT(DefaultRate_Pct, '%')                    AS DefaultRate,
    CONCAT('$', FORMAT_NUMBER(TotalOutstanding, 0)) AS Outstanding
FROM gold_branch_performance
ORDER BY TotalLoanVolume DESC
LIMIT 5

In [ ]:
%%sql
-- All Gold tables created
-- Confirm that all gold aggregation tables were created and populated with rows.
SELECT 'gold_branch_performance' AS GoldTable, COUNT(*) AS Rows FROM gold_branch_performance UNION ALL
SELECT 'gold_monthly_trends',                  COUNT(*)         FROM gold_monthly_trends      UNION ALL
SELECT 'gold_customer_segments',               COUNT(*)         FROM gold_customer_segments   UNION ALL
SELECT 'gold_product_performance',             COUNT(*)         FROM gold_product_performance